# Initialize

In [ ]:
# @title Global Variables

from google.colab import drive
drive.mount('/content/drive')

!pip install scikit-learn -q
!pip install sqlitedict -q
!pip install openai -q

import os
from sqlitedict import SqliteDict
import hashlib
import json

working_dir = "/content/drive/MyDrive/LLM Auction/"
dataset_dir = "/content/drive/MyDrive/LLM Auction/dataset/"
logprobs_cache_dir = working_dir + "cache_logprobs/"
hfmodels_cache_dir = working_dir + "cache_hfmodels/"

current_directory = os.getcwd()
print(f"current_directory: {current_directory}")

os.makedirs(logprobs_cache_dir, exist_ok=True)
os.makedirs(hfmodels_cache_dir, exist_ok=True)

# Cache key function for caching the output
def generate_cache_key(params):
    '''
    # generate a unique cache key based on the request parameters
    ## md5 seems to be enough
    params: dict, request parameters
    '''
    params_string = json.dumps(params, sort_keys=True)
    return hashlib.sha256(params_string.encode('utf-8')).hexdigest()


Mounted at /content/drive
  Preparing metadata (setup.py) ... done
current_directory: /content


In [ ]:
# @title Utils

def process_corpus(text):
    lines = text.strip().split('\n')
    clean_lines = []
    ad_slot_positions = []

    for index, line in enumerate(lines):
        if line.strip().lower() == '[ad slot]':
            ad_slot_positions.append(len(clean_lines))
        else:
            clean_lines.append(line)

    return clean_lines, ad_slot_positions


def insert_ad(lines, ad_slot_position, ad):
    merged = lines.copy()

    merged.insert(ad_slot_position, ad)

    return merged


def insert_ads(lines, ad_slot_positions, ads):
    if len(ad_slot_positions) != len(ads):
        raise ValueError("ad_slot_positions and ads must be the same length.")

    merged = lines.copy()

    # Zip positions with ads, sort by position descending
    insertions = sorted(zip(ad_slot_positions, ads), reverse=True)

    for pos, ad in insertions:
        merged.insert(pos, ad)

    return merged

def extract_json(text):
    match = re.search(r'(\{.*\})', text, re.DOTALL)
    if match:
        json_str = match.group(1)
        return json_str
    else:
        print("Can not find JSON")
        return None

# Sentence-embedding

In [ ]:
# @title Define the EmbeddingBasedScoreCalculator Class

from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

class EmbeddingBasedScoreCalculator:
    def __init__(self, model_name="Qwen/Qwen3-Embedding-8B", cache_dir="./cache_embedding/", debug_mode=1):
        self.model_name = model_name
        self.debug_mode = debug_mode

        # Initialize sentence embedding model
        try:
            self.sentence_embedding_model = SentenceTransformer(model_name, trust_remote_code=True).cuda()
        except Exception as e:
            print(f"Warning: Could not load CUDA model for {model_name}. Trying CPU.")
            try:
                self.sentence_embedding_model = SentenceTransformer(model_name, trust_remote_code=True)
            except Exception as e_cpu:
                print(f"Error: Could not load CPU model for {model_name}. Embedding based scoring will not work.")
                self.sentence_embedding_model = None

        # Initialize cache
        os.makedirs(cache_dir, exist_ok=True)
        cache_filename = f"cos_sim_{(model_name).replace('/', '--')}.sqlite"
        self.cos_sim_db = SqliteDict(os.path.join(cache_dir, cache_filename), autocommit=True)

    def generate_cache_key(self, params):
        '''
        # generate a unique cache key based on the request parameters
        ## md5 seems to be enough
        params: dict, request parameters
        '''
        params_string = json.dumps(params, sort_keys=True)
        return hashlib.sha256(params_string.encode('utf-8')).hexdigest()


    def get_cos_sim(self, source, target, use_cache=True):
        if self.sentence_embedding_model is None:
            print("Embedding model not loaded. Cannot calculate cosine similarity.")
            return 0.0 # Return a default value or handle as appropriate

        params = {
            "query": "query_cos_sim",
            "model": self.model_name,
            "source": source,
            "target": target,
        }
        key = self.generate_cache_key(params)
        if use_cache and (key in self.cos_sim_db):
            if self.debug_mode > 1: print(f"Cache hit for cos_sim key: {key}")
            return self.cos_sim_db[key]
        if self.debug_mode > 1: print(f"Cache miss for cos_sim key: {key}")

        source_embeddings = self.sentence_embedding_model.encode(source, convert_to_tensor=True)
        target_embeddings = self.sentence_embedding_model.encode(target, convert_to_tensor=True)

        similarity = cos_sim(source_embeddings, target_embeddings).item() # Get the scalar value

        if use_cache:
            self.cos_sim_db[key] = similarity

        return similarity

    def split_sentences(self, text):
        # Simple split by newline for now, can be extended with more robust sentence splitting
        sentences = text.strip().split("\n")
        sentences = [sentence.strip() for sentence in sentences if sentence.strip()]
        # Add empty strings at the beginning and end to consider coherence with implicit start/end
        sentences = [""] + sentences + [""]
        return sentences

    def calculate_score(self, response_text):
        """
        Calculates an embedding-based coherence score for a given response text.
        The score is the sum of cosine similarities between consecutive "sentences"
        (lines in this implementation).
        """
        if self.sentence_embedding_model is None:
             print("Embedding model not loaded. Cannot calculate score.")
             return 0.0 # Return a default value or handle as appropriate


        sentences = self.split_sentences(response_text)

        score = 0.0
        for i in range(len(sentences) - 1):
            score += self.get_cos_sim(sentences[i], sentences[i+1])

        if self.debug_mode > 0:
            # print(f"Response Text:\n{response_text}")
            # print(f"Split Sentences: {sentences}")
            print(f"Calculated Embedding Score: {score}")

        return score

    def process_corpus(self, text):
        lines = text.strip().split('\n')
        clean_lines = []
        ad_slot_positions = []

        for index, line in enumerate(lines):
            if line.strip().lower() == '[ad slot]':
                ad_slot_positions.append(len(clean_lines))
            else:
                clean_lines.append(line)

        return clean_lines, ad_slot_positions

    def insert_ad(self, lines, ad_slot_position, ad):
        merged = lines.copy()
        merged.insert(ad_slot_position, ad)
        return merged

    def insert_ads(self, lines, ad_slot_positions, ads):
        if len(ad_slot_positions) != len(ads):
            raise ValueError("ad_slot_positions and ads must be the same length.")

        merged = lines.copy()

        # Zip positions with ads, sort by position descending
        insertions = sorted(zip(ad_slot_positions, ads), reverse=True)

        for pos, ad in insertions:
            merged.insert(pos, ad)

        return merged


    def calculate_coherence(self, response_with_slots, ads_to_insert):
        """
        Calculates the embedding-based coherence score difference by inserting ads
        into specified slots and comparing scores.

        Args:
            response_with_slots (str): The original LLM response containing [Ad Slot] markers.
            ads_to_insert (list of tuples): A list of (ad_genre, slot_index) tuples,
                                           where slot_index is 0-based.

        Returns:
            float: The coherence score (embedding score difference).
        """
        lines, ad_slot_positions = self.process_corpus(response_with_slots)
        original_response = "\n".join(lines)

        # Calculate embedding score of the original response without ads
        original_embedding_score = self.calculate_score(original_response)

        # Prepare ads and their target positions
        ads = []
        insertion_positions = []
        for ad_genre, slot_index in ads_to_insert:
            # Using ad_genre as the ad text formatted with [Ad: ]
            ad_text = f"Ad: {ad_genre}"
            ads.append(ad_text)
            insertion_positions.append(ad_slot_positions[slot_index])

        # Insert the ads into the response
        response_with_inserted_ads = "\n".join(self.insert_ads(lines, insertion_positions, ads))
        # print(response_with_inserted_ads)

        # Calculate embedding score of the response with inserted ads
        response_with_ads_embedding_score = self.calculate_score(response_with_inserted_ads)

        # Coherence score is the embedding score difference
        coherence_score = response_with_ads_embedding_score - original_embedding_score

        if self.debug_mode > 0:
            print(f"Original Response:\n{original_response}")
            print(f"Original Embedding Score: {original_embedding_score}")
            print(f"Response with Ads:\n{response_with_inserted_ads}")
            print(f"Embedding Score with Ads: {response_with_ads_embedding_score}")
            print(f"Coherence Score (Embedding Diff): {coherence_score}")

        return coherence_score




# LMExaminer

In [ ]:
# @title Define the LMExaminerCalculator Class

import openai
import threading
import re

class LMExaminerCalculator:
    def __init__(self, model_name="deepseek/deepseek-r1", cache_dir="./cache_lmexaminer/", debug_mode=1):
        self.model_name = model_name
        self.debug_mode = debug_mode

        # Initialize the API client (using OpenRouter as in your code)
        try:
            self.openrouter_client = openai.OpenAI(
                base_url="https://openrouter.ai/api/v1",
                api_key="" # Replace with your actual key if different
            )
            self.openrouter_chat = self.openrouter_client.chat.completions
        except Exception as e:
            print(f"Error initializing OpenAI client for LMExaminer: {e}")
            self.openrouter_client = None
            self.openrouter_chat = None


        # Initialize cache
        os.makedirs(cache_dir, exist_ok=True)
        cache_filename = f"lmexaminer_{(model_name).replace('/', '--')}.sqlite"
        self.lmexaminer_db = SqliteDict(os.path.join(cache_dir, cache_filename), autocommit=True)
        self.cache_lock = threading.Lock() # Lock for safe cache operations

        self.system_prompt = """
You are an expert in digital advertising and user experience. Your task is to rate how suitable different ad genres would be if inserted into a specific location within an LLM-generated response.
"""

        self.user_prompt_template = """
Context: {}
User Query: {}

The LLM response contains an ad slot (marked as [Ad Slot]) where an advertisement could be inserted.

Here is the text surrounding Ad Slot:

{}
[Ad Slot] (THIS IS WHERE THE AD WOULD BE INSERTED)
{}

For each of the following ad genres, rate the suitability of inserting an ad from that genre into this slot on a scale from 1 (poor) to 5 (excellent).

When rating, consider:
- Fluency: Would the ad fit naturally within the flow of the text?
- Coherence: Would the ad align logically with the context of the response?

Here are the ad genres to rate:

Ad Genres:
1. Airlines - Examples: Flight deals, airline promotions
2. Apparel - Examples: Clothing, shoes, accessories
3. Automotive - Examples: Cars, motorcycles, EV vehicles
4. Electronics - Examples: Smartphones, computers, tablets
5. FMCG (Fast-Moving Consumer Goods) - Examples: Personal care products, household items
6. Finance - Examples: Banking services, insurance, credit cards
7. Hotels - Examples: Hotel chains, booking services
8. Media - Examples: Streaming services, social media platforms
9. Packaged Food - Examples: Snacks, beverages, prepared meals
10. Restaurants - Examples: Fast food, cafes, dining establishments

For each genre, provide:
1. A rating from 1-5
2. A brief explanation for your rating

Format your response as JSON with this structure:
{{
  "ratings": {{
    "Airlines": {{ "explanation": "...", "score": X }},
    "Apparel": {{ "explanation": "...", "score": X }},
    ...
  }}
}}

Ensure your ratings are justified based on the context and the natural flow of the text.
"""

    def generate_cache_key(self, params):
        '''
        # generate a unique cache key based on the request parameters
        ## md5 seems to be enough
        params: dict, request parameters
        '''
        params_string = json.dumps(params, sort_keys=True)
        return hashlib.sha256(params_string.encode('utf-8')).hexdigest()

    def call_api(self, params, use_cache=True):
        if self.openrouter_chat is None:
            print("API client not initialized. Cannot make API call.")
            return None

        key = self.generate_cache_key(params)
        with self.cache_lock:
            if use_cache and (key in self.lmexaminer_db):
                if self.debug_mode > 1: print(f"Cache hit for LMExaminer key: {key}")
                return self.lmexaminer_db[key]
        if self.debug_mode > 1: print(f"Cache miss for LMExaminer key: {key}")


        try:
            response = self.openrouter_chat.create(**params)
            if response.choices is None:
                print(f"Error calling API: {response.error['message']}")
                return None
            content = response

            with self.cache_lock:
                if use_cache:
                    self.lmexaminer_db[key] = content
            return content
        except Exception as e:
            print(f"An error occurred during API call: {e}")
            return None

    def simple_call_api(self, system_text, user_text, max_tokens=4000, temperature=0):
        messages = [{
            "role": "system",
            "content": system_text,
        }, {
            "role": "user",
            "content": user_text,
        }] if system_text != "" else [{
            "role": "user",
            "content": user_text,
        }]
        params = {
            "model": self.model_name,
            "messages": messages,
            "temperature": temperature,
            "max_tokens": max_tokens,
            "frequency_penalty": 0,
            "presence_penalty": 0
        }
        return self.call_api(params)

    def extract_json(self, text):
        match = re.search(r'(\{.*\})', text, re.DOTALL)
        if match:
            json_str = match.group(1)
            return json.loads(json_str)
        else:
            if self.debug_mode > 0: print("Warning: Could not find JSON in LMExaminer response.")
            return None


    def get_suitability_ratings(self, context, user_query, response_with_slots, slot_index):
        """
        Gets suitability ratings for all ad genres from the LM-examiner for a specific ad slot.

        Args:
            context (str): The overall context of the interaction.
            user_query (str): The user's original query.
            response_with_slots (str): The LLM response containing [Ad Slot] markers.
            slot_index (int): The 0-based index of the ad slot to examine.

        Returns:
            dict: A dictionary where keys are ad genres and values are dictionaries
                  containing 'explanation' and 'score', or None if API call or JSON extraction fails.
        """
        if self.openrouter_client is None:
             print("LMExaminer API client not initialized. Cannot get ratings.")
             return None

        # Use the process_corpus from your utils or adapt it here
        lines, ad_slot_positions = process_corpus(response_with_slots)

        if slot_index < 0 or slot_index >= len(ad_slot_positions):
            print(f"Error: slot_index {slot_index} is out of bounds for {len(ad_slot_positions)} ad slots.")
            return None

        target_pos_in_clean_lines = ad_slot_positions[slot_index]

        text_before_ad_slot = "\n".join(lines[:target_pos_in_clean_lines])
        text_after_ad_slot = "\n".join(lines[target_pos_in_clean_lines:])


        user_prompt_formatted = self.user_prompt_template.format(
            context,
            user_query,
            text_before_ad_slot,
            text_after_ad_slot
        )

        api_response_text = self.simple_call_api(
            self.system_prompt,
            user_prompt_formatted,
        )

        if api_response_text:
            json_data = self.extract_json(api_response_text.choices[0].message.content)
            if json_data and "ratings" in json_data:
                return json_data["ratings"]
            else:
                if self.debug_mode > 0: print("Could not extract valid JSON ratings from API response.")
                return None
        else:
            if self.debug_mode > 0: print("API call returned no response text.")
            return None


    def calculate_coherence(self, context, user_query, response_with_slots, ads_to_insert):
        """
        Calculates a combined coherence score based on LM-examiner ratings for the inserted ads.
        This method assumes the score is the sum of the suitability ratings for the
        specific ad genres inserted at their respective locations.

        Args:
            context (str): The overall context of the interaction.
            user_query (str): The user's original query.
            response_with_slots (str): The LLM response containing [Ad Slot] markers.
            ads_to_insert (list of tuples): A list of (ad_genre, slot_index) tuples,
                                           where slot_index is 0-based.

        Returns:
            float: The combined LM-examiner suitability score for the inserted ads,
                   or None if ratings cannot be retrieved for any of the insertions.
        """
        total_suitability_score = 0.0
        processed_slots = set()

        for ad_genre, slot_index in ads_to_insert:
            # Only call the examiner once per slot if multiple ads target the same slot
            if slot_index not in processed_slots:
                suitability_ratings = self.get_suitability_ratings(
                    context,
                    user_query,
                    response_with_slots,
                    slot_index
                )
                processed_slots.add(slot_index)

                if suitability_ratings is None:
                    if self.debug_mode > 0:
                        print(f"Warning: Could not retrieve ratings for slot index {slot_index}. Skipping this insertion.")
                    # Depending on desired behavior, you might return None immediately or handle this differently
                    continue # Skip this specific insertion, continue with others if possible

            # Find the score for the specific ad_genre at this slot
            if suitability_ratings and ad_genre in suitability_ratings:
                score = suitability_ratings[ad_genre].get("score", 0.0) # Use .get with default 0.0 in case score is missing
                total_suitability_score += score
                if self.debug_mode > 0:
                    print(f"Added score {score} for '{ad_genre}' at slot {slot_index}")
            else:
                 if self.debug_mode > 0:
                     print(f"Warning: Could not find rating for ad genre '{ad_genre}' in slot {slot_index} ratings.")


        if self.debug_mode > 0:
            print(f"Total LM-Examiner Suitability Score: {total_suitability_score}")

        return total_suitability_score

# Experiments

In [ ]:
# @title Experiment: Comparing Calculator Scores Across All Data
import csv

file_path = dataset_dir + 'data.csv'

data = []
with open(file_path, mode='r', encoding='utf-8') as file:
    csv_reader = csv.DictReader(file)  # Automatically uses the first row as keys
    for row in csv_reader:
        data.append(row)


results = {}

In [ ]:
ad_genre = '''Airlines
Apparel
Automotive
Electronics
Fast-Moving Consumer Goods (FMCG)
Finance
Hotels
Media
Packaged Food (and Beverage)
Restaurants'''

ad_genre_list = ad_genre.split("\n")

# # Logprob Calculation (ensure this part uses coherence_calculator.calculate_coherence)
# print("Calculating Logprob Coherence scores...")

# prompts = {}
# prompts["logprob"] = {}

# prompts["logprob"]["system"] = '''You are an intelligent and helpful assistant who provides responses based on user requests. You may add an ad in your response that fits well in the near context with the format [Ad: ad content]. You can choose the ad from the following categories:
# 1.	Airlines
# 2.	Apparel – Clothing, shoes, hats, facial coverings, accessories, and other wearable items.
# 3.	Automotive – Cars, motorcycles, electric vehicles, or related products/services.
# 4.	Electronics – Smartphones, computers, earphones, tablets, TVs, etc.
# 5.	Fast-Moving Consumer Goods (FMCG) – Personal care products (e.g., shampoo, toothpaste, soap), cosmetics, over-the-counter (OTC) drugs, household items (detergent, dish soap, paper towels, toilet paper, trash bags), etc.
# 6.	Finance – Banks, insurance companies, investment services, credit cards, retirement planning, loans (including Baitiao/Huabei), etc.
# 7.	Hotels
# 8.	Media – Platforms or services such as Netflix, ESPN, YouTube, X (formerly Twitter), BiliBili, Tencent Video, etc.
# 9.	Packaged Food (and Beverage) – Potato chips, bread, crackers, cereal, pasta, canned food, frozen meals, sauces, soft drinks, etc.
# 10.	Restaurants – including cafes, fast food
# '''

# results["logprob"] = {}

# logprob_coherence_calculator = LogprobCoherenceCalculator(logprobs_calculator=logprobs_calculator, debug_mode=debug_mode)

# for task_id, entry in enumerate(data):
#     system_prompt = prompts["logprob"]["system"]
#     user_prompt = entry["Prompt"]
#     response_with_slots = entry["Response"]
#     _, ad_slot_positions = process_corpus(response_with_slots)

#     if not ad_slot_positions: continue # Skip if no ad slots

#     for slot_index in range(len(ad_slot_positions)):
#         for ad_id, ad_genre in enumerate(ad_genre_list):
#              # For this calculation, we just insert one ad of this genre at this slot
#             ads_to_insert = [(ad_genre, slot_index)]
#             coherence_score = logprob_coherence_calculator.calculate_coherence(
#                 system_prompt,
#                 user_prompt,
#                 response_with_slots,
#                 ads_to_insert
#             )
#             # Using 1-based indexing for task_id and slot_id for the results key
#             results["logprob"][f"T{task_id+1:02d}A{slot_index+1:02d}.{ad_id}"] = coherence_score


# Embedding Calculation (ensure this part uses embedding_based_score_calculator.calculate_coherence)


model_names = {
    "qwen3-8b": "Qwen/Qwen3-Embedding-8B",
    "qwen3-4b": "Qwen/Qwen3-Embedding-4B",
    "qwen3-0.6b": "Qwen/Qwen3-Embedding-0.6B",
}

model_list = ["qwen3-8b", "qwen3-4b", "qwen3-0.6b"] #["qwen3-8b", "qwen3-4b", "qwen3-0.6b"]


for judge_short in model_list:
    model_name = model_names[judge_short]
    print(f"Calculating Embedding Coherence scores for {model_name}...")

    results[f"embedding-{judge_short}"] = {}

    embedding_based_score_calculator = EmbeddingBasedScoreCalculator(cache_dir=logprobs_cache_dir, model_name = model_name, debug_mode=debug_mode)

    for task_id, entry in enumerate(data):
        user_prompt = entry["Prompt"] # Not used in embedding coherence calculation
        response_with_slots = entry["Response"]
        _, ad_slot_positions = process_corpus(response_with_slots)

        if not ad_slot_positions: continue # Skip if no ad slots

        for slot_index in range(len(ad_slot_positions)):
            for ad_id, ad_genre in enumerate(ad_genre_list):
                ads_to_insert = [(ad_genre, slot_index)]
                coherence_score = embedding_based_score_calculator.calculate_coherence(
                    response_with_slots,
                    ads_to_insert
                )
                # Using 1-based indexing for task_id and slot_id for the results key
                results[f"embedding-{judge_short}"][f"T{task_id+1:02d}A{slot_index+1:02d}.{ad_id}"] = coherence_score


Calculating Embedding Coherence scores for Qwen/Qwen3-Embedding-8B...


NameError: name 'EmbeddingBasedScoreCalculator' is not defined

In [ ]:

ad_genre = '''Airlines
Apparel
Automotive
Electronics
FMCG
Finance
Hotels
Media
Packaged Food
Restaurants'''

ad_genre_list = ad_genre.split("\n")

ad_genre_to_id = {genre: idx for idx, genre in enumerate(ad_genre_list)}
ad_genre_to_id["FMCG (Fast-Moving Consumer Goods)"] = 4
ad_genre_to_id["Fast-Moving Consumer Goods"] = 4

# LM-Examiner Calculation (ensure this part uses lmexaminer_calculator.get_suitability_ratings)

model_names = {
    "gpt5": "openai/gpt-5",
    "gpt4omini": "openai/gpt-4o-mini",
    "gpt4o": "openai/gpt-4o",
    "gpt4turbo": "openai/gpt-4-turbo",
    "gpt35turbo": "openai/gpt-3.5-turbo-0125",
    "claude2": "anthropic/claude-2",
    "claude3haiku": "anthropic/claude-3-haiku",
    "claude3sonnet": "anthropic/claude-3-sonnet",
    "claude3opus": "anthropic/claude-3-opus",
    "geminipro1": "google/gemini-pro",
    "geminipro15": "google/gemini-pro-1.5",
    "llama2-70b": "meta-llama/llama-2-70b-chat",
    "llama3-8b": "meta-llama/llama-3-8b-instruct",
    "llama3-70b": "meta-llama/llama-3-70b-instruct",
    "mixtral-7b": "mistralai/mistral-7b-instruct:nitro",
    "mixtral-8x7b": "mistralai/mixtral-8x7b-instruct",
    "mixtral-8x22b": "mistralai/mixtral-8x22b-instruct",
    "wizardlm2-7b": "microsoft/wizardlm-2-8x7b",
    "wizardlm2-8x22b": "microsoft/wizardlm-2-8x22b",
    "o1": "openai/o1-preview",
    "o1mini": "openai/o1-mini",
    "llama31-8b": "meta-llama/llama-3.1-8b-instruct",
    "llama31-70b": "meta-llama/llama-3.1-70b-instruct",
    "llama31-405b": "meta-llama/llama-3.1-405b-instruct",
    "mistral-large": "mistralai/mistral-large",
    "mistral-small": "mistralai/mistral-small",
    "mistral-tiny": "mistralai/mistral-tiny",
    "deepseek-v3": "deepseek/deepseek-chat",
    "deepseek-r1": "deepseek/deepseek-r1",
    "qwen3-4b": "qwen/qwen3-4b",
    "qwen3-8b": "qwen/qwen3-8b",
    "qwen3-32b": "qwen/qwen3-32b",
}

model_list = ["qwen3-8b", "qwen3-32b", "deepseek-v3","deepseek-r1","gpt4omini","gpt4o","gpt5"]

for judge_short in model_list:
    model_name = model_names[judge_short]
    print(f"Calculating LM-Examiner scores for {model_name}...")

    lmexaminer_calculator = LMExaminerCalculator(model_name=model_name, cache_dir=logprobs_cache_dir)

    results[f"lmexaminer-v2-{judge_short}"] = {}
    for task_id, entry in enumerate(data, start=1):
        context = entry["Context"] # Assuming Context is available
        user_prompt = entry["Prompt"]
        response_with_slots = entry["Response"]
        lines, ad_slot_pos = process_corpus(response_with_slots)

        if not ad_slot_pos: continue # Skip if no ad slots

        for pos_id, pos in enumerate(ad_slot_pos, start=1):
            slot_index = pos_id - 1 # 0-based index for the method

            # Get suitability ratings for all ad genres at this slot
            suitability_ratings = lmexaminer_calculator.get_suitability_ratings(
                context,
                user_prompt,
                response_with_slots,
                slot_index
            )

            if suitability_ratings:
                # Store the score for each ad genre at this position

                for ad_genre, details in suitability_ratings.items():
                    if ad_genre in ad_genre_to_id:
                        ad_id = ad_genre_to_id[ad_genre]
                        results[f"lmexaminer-v2-{judge_short}"][f"T{task_id:02d}A{pos_id:02d}.{ad_id}"] = details.get("score", "")
                    else:
                        if debug_mode > 0:
                            print(f"Warning: LM-Examiner returned rating for unknown genre '{ad_genre}' in Task {task_id}, Slot {pos_id}")

            else:
                if debug_mode > 0:
                    print(f"Could not get LM-examiner ratings for Task {task_id}, Slot {pos_id}")


print("\nFinished calculating scores for Experiment 3.")
print("Results dictionary is populated and ready for analysis/saving.")



Calculating LM-Examiner scores for qwen/qwen3-8b...


# Save the results

In [ ]:
csv_path = os.path.join(dataset_dir, "scorer_results.csv")
data, columns = {}, set()

# Load existing CSV (if any)
if os.path.exists(csv_path) and os.path.getsize(csv_path) > 0:
    with open(csv_path, newline="") as f:
        r = csv.reader(f)
        header = next(r, None)
        if header:
            old_cols = header[1:]
            columns.update(old_cols)
            for row in r:
                if not row: continue
                key = row[0]
                data.setdefault(key, {}).update({c: v for c, v in zip(old_cols, row[1:])})

# Merge results: update existing keys, append new keys; track all columns
for k, sub in results.items():
    data.setdefault(k, {}).update(sub)
    columns.update(sub.keys())

cols = sorted(columns)
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow([""] + cols)
    for k, sub in data.items():  # preserves existing order; new keys go at the end
        w.writerow([k] + [sub.get(c, "") for c in cols])
